In [1]:
!pip install -q transformers accelerate qwen-vl-utils gradio sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 14.6 MB/s eta 0:00:00


In [2]:
import torch, json, re
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(
    MODEL, min_pixels=256*28*28, max_pixels=768*28*28  # keeps memory low
)

def vlm(image_path, prompt, max_new_tokens=700):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image_path},
        {"type": "text", "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(messages)
    inputs = processor(text=[text], images=imgs, videos=vids,
                       padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
def extract_text(image_path):
    return vlm(image_path, "Transcribe all the text on this page exactly. Keep Urdu and English as written.")

def make_quiz(image_path, n=3, language="English"):
    prompt = f"""Read this study material and write {n} multiple-choice questions in {language}.
Return ONLY valid JSON, a list of objects with keys:
"question", "options" (4 strings), "answer" (one of the options), "explanation"."""
    raw = vlm(image_path, prompt, max_new_tokens=900)
    match = re.search(r"\[.*\]", raw, re.S)
    try:
        return json.loads(match.group(0))
    except Exception:
        return {"error": "Could not parse JSON", "raw": raw}

In [4]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
chunks, index = [], None

def build_index(text, size=300):
    global chunks, index
    words = text.split()
    chunks = [" ".join(words[i:i+size]) for i in range(0, len(words), size)] or [text]
    emb = embedder.encode(chunks, normalize_embeddings=True)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.array(emb, dtype="float32"))

def answer(question, image_path, k=2):
    q = embedder.encode([question], normalize_embeddings=True)
    _, ids = index.search(np.array(q, dtype="float32"), k)
    context = "\n".join(chunks[i] for i in ids[0])
    return vlm(image_path, f"Using only this context:\n{context}\n\nAnswer the question: {question}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
import gradio as gr

def run(image_path, mode, language, question):
    if mode == "Generate quiz":
        return json.dumps(make_quiz(image_path, 3, language), indent=2, ensure_ascii=False)
    build_index(extract_text(image_path))
    return answer(question, image_path)

gr.Interface(
    fn=run,
    inputs=[gr.Image(type="filepath"),
            gr.Radio(["Generate quiz", "Ask a question"], value="Generate quiz"),
            gr.Radio(["English", "Urdu"], value="English"),
            gr.Textbox(label="Question (for 'Ask a question' mode)")],
    outputs=gr.Textbox(label="Result", lines=18),
    title="Visual Study Assistant",
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://249b963ebb5b98b43e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
